# Equivalencia contra el oráculo


Los resultados de este repositorio contra `oraculo_v4/`: las mismas tablas
calculadas por estos mismos notebooks antes del recorte de la base y de la
codificación a enteros.

La comparación es informativa porque los cambios no afectan a todo por igual:

- `analiceDB/` describe la base y sus componentes conexas, y el recorte **debe** moverlo;
- `genome_prioritization/` y `huerfanas/` trabajan sobre la capa de anotaciones y
  los blancos druggables, y **no** deberían moverse.

Lo que debe cambiar cambia de forma explicable, y lo que no debe cambiar no cambia.

Salidas: `10_equivalencia.csv`, `10_discrepancias.csv`.


## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
# raiz del repositorio: se busca hacia arriba la carpeta que tiene DB/,
# asi el notebook corre desde donde sea que se haya clonado
RAIZ = Path.cwd()
while not (RAIZ / "DB").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "comun"))
sys.path.insert(0, str(RAIZ / "verificacion"))

import tdr
import funciones_verificacion as fv

SALIDAS = RAIZ / "resultados"
ORACULO = RAIZ / "oraculo_v4"
NB      = "10"
DESTINO = tdr.SALIDAS / "verificacion_out"
(DESTINO / "figuras").mkdir(parents=True, exist_ok=True)
plt = tdr.estilo()
DESTINO

## Datos

In [ ]:
# Que hay de cada lado, antes de comparar nada.
inv = []
for lado, base in (("nuevo", SALIDAS), ("oraculo", ORACULO)):
    for sub in sorted(base.glob("*_out")):
        for f in sorted(sub.glob("*.csv")):
            inv.append({"lado": lado, "analisis": sub.name.replace("_out", ""),
                        "tabla": f.name, "filas": sum(1 for _ in open(f)) - 1})
inv = pd.DataFrame(inv)
inv.pivot_table(index="analisis", columns="lado", values="tabla", aggfunc="count", fill_value=0)

## Acondicionamiento

In [ ]:
# Las tablas por especie se nombran con el codigo (01_26_...) de un lado y con
# el prefijo sp (01_sp26_...) del otro: se emparejan normalizando el nombre.
import re
def normalizar(n): return re.sub(r"(?<=_)sp(?=\d)", "", n)

pares = {}
for sub in sorted(ORACULO.glob("*_out")):
    for v in sorted(sub.glob("*.csv")):
        cand = SALIDAS / sub.name / v.name
        if not cand.exists():
            cand = SALIDAS / sub.name / normalizar(v.name)
        pares[str(v)] = cand if cand.exists() else None
faltan = [k for k, v in pares.items() if v is None]
print(f"{len(pares)} tablas del oraculo | sin par nuevo: {len(faltan)}")
for f in faltan[:12]: print("   ", Path(f).name)

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
filas = []
for viejo, nuevo in pares.items():
    viejo = Path(viejo); analisis = viejo.parent.name.replace("_out", "")
    if nuevo is None:
        filas.append({"analisis": analisis, "tabla": viejo.name, "columna": "-",
                      "estado": "sin par nuevo"})
        continue
    dn, dv = pd.read_csv(nuevo), pd.read_csv(viejo)
    cmp = fv.comparar_tabla(dn, dv, fv.CLAVES.get(viejo.name, "especie"))
    for _, r in cmp.iterrows():
        filas.append({"analisis": analisis, "tabla": viejo.name, "columna": r["columna"],
                      "n": r["n"], "pct_iguales": r["iguales"], "dmax": r["dmax"],
                      "estado": fv.clasificar(r["iguales"], analisis),
                      "filas_nueva": cmp.attrs["filas_nueva"],
                      "filas_vieja": cmp.attrs["filas_vieja"]})
    if cmp.empty:
        filas.append({"analisis": analisis, "tabla": viejo.name,
                      "columna": "(sin columnas numericas comunes)", "estado": "sin datos",
                      "filas_nueva": cmp.attrs["filas_nueva"],
                      "filas_vieja": cmp.attrs["filas_vieja"]})

eq = pd.DataFrame(filas)
eq.to_csv(DESTINO / f"{NB}_equivalencia.csv", index=False)
eq[eq["estado"] == "DISCREPANCIA"].to_csv(DESTINO / f"{NB}_discrepancias.csv", index=False)

tdr.escribir_meta(DESTINO, NB, notebook="10_equivalencia.ipynb",
                  params={"rtol": fv.RTOL, "esperado": fv.ESPERADO},
                  tablas_comparadas=len(pares))
eq["estado"].value_counts()

# Resultados

In [ ]:
eq = pd.read_csv(DESTINO / f"{NB}_equivalencia.csv")
resumen = (eq.groupby(["analisis", "estado"]).size().unstack(fill_value=0))
resumen

In [ ]:
# Las discrepancias, que son lo unico que obliga a investigar.
disc = eq[eq["estado"] == "DISCREPANCIA"]
print(f"{len(disc)} columnas con discrepancia")
disc.sort_values("dmax", ascending=False).head(30)

In [ ]:
# Lo que cambio en analiceDB, que es donde el recorte tiene que verse.
cam = eq[(eq["analisis"] == "analiceDB") & (eq["estado"] != "idéntico")]
cam[["tabla", "columna", "n", "pct_iguales", "dmax", "filas_nueva", "filas_vieja"]].head(30)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2), tight_layout=True)
orden = ["idéntico", "explicado por el recorte", "DISCREPANCIA", "sin datos", "sin par nuevo"]
col = {"idéntico": tdr.ST_GOOD, "explicado por el recorte": tdr.S4,
       "DISCREPANCIA": tdr.ST_CRIT, "sin datos": tdr.MUTED, "sin par nuevo": tdr.INK2}
piv = eq.groupby(["analisis", "estado"]).size().unstack(fill_value=0)
piv = piv.reindex(columns=[c for c in orden if c in piv.columns])
abajo = np.zeros(len(piv))
for c in piv.columns:
    ax.barh(range(len(piv)), piv[c], left=abajo, color=col[c], label=c)
    abajo += piv[c].values
ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
ax.set_xlabel("columnas comparadas")
ax.set_title("Equivalencia contra el oráculo, por análisis")
ax.legend(fontsize=7, loc="lower right")
tdr.guardar(fig, f"{NB}_f01_equivalencia", DESTINO / "figuras")